In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import hashlib
import gzip
import numpy as np
import re
import os
from pandas.api.types import CategoricalDtype
from matplotlib.lines import Line2D

In [ ]:
def extractInfo(file_name):
    # Updated regular expression to extract components
    pattern = r"(?P<sample>\w+\.\w+)\.raw-raw\.rna\.(?P<assembler>\w+)\.(?P<config>\w+)\.tsv"
    
    # Match the pattern
    match = re.match(pattern, file_name)
    
    # Extract components if the pattern matches
    if match:
        sample = match.group("sample")
        assembler = match.group("assembler")
        config = match.group("config")
        return sample, assembler, config
    return None, None, None

In [ ]:
def curate_df(path_tsv, cois):
    files = glob.glob(path_tsv)
    dfs = []

    for file in files:
        # Check if the file matches any configuration of interest
        if any(conf in file for conf in cois):
            # Read the file into a DataFrame
            df = pd.read_csv(file, sep='\t', names=[
                "query", "target", "seq.Id.", "alnLen", "MM", "gaps", 
                "startQuery", "EndQuery", "startTarget", "EndTarget", 
                "Eval", "bit score", "queryLen", "targetLen", 
                "queryCov", "targetCov"
            ])

            # Extract components from the filename
            name = os.path.basename(file)
            sample, assembler, config = extractInfo(name)

            # Map the assembler to its "nice" version using the mapping dictionary
            assembler_mapped = mapping.get(f"{assembler}.{config}", assembler)

            # Add extracted and mapped information as new columns
            df["sample"] = sample
            df["assembler"] = assembler_mapped
            df["config"] = config

            dfs.append(df)

    # Concatenate all DataFrames into a single DataFrame
    big_df = pd.concat(dfs, ignore_index=True)

    return big_df


In [ ]:
configsOfInterest = ['megahit.config10', 'carpedeam.configSafe0', 'carpedeam.configUnsafe0', 'penguin.config10', 'spades.config10'] 

In [ ]:
# Mapping dictionary for assembler names
mapping = {
    "carpedeam.configUnsafe0": "CarpeDeam\n(unsafe mode)",
    "carpedeam.configSafe0": "CarpeDeam\n(safe mode)",
    "penguin.config10": "PenguiN",
    "megahit.config10": "MEGAHIT",
    "spades.config10": "metaSPAdes",
}

In [ ]:
path="data/empirical/results/assembly-rna/silva/*tsv"

In [ ]:
revi = curate_df(path, configsOfInterest)

In [ ]:
def filter_bacteria_identifiers(file_path, df):
    # Read the input file
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Extract identifiers for rows containing "Bacteria"
    identifiers = []
    for line in lines:
        if "Bacteria" in line:
            identifiers.append(line.split()[0])  # Extract the identifier (first element in the line)
    
    # Filter the DataFrame on these identifiers
    filtered_df = df[df['target'].apply(lambda x: any(identifier in x for identifier in identifiers))]

    return filtered_df


In [ ]:
def filter_targets_from_file(df, file_path):
    # Step 1: Get the set of unique targets from the DataFrame
    target_set = set(df['target'])

    # Step 2: Create a set of identifiers from the input file, cleaning up '\x00'
    file_set = set()
    with open(file_path, 'r') as f:
        for line in f:
            identifier = line.split()[0].replace('\x00', '')  # Remove '\x00' from the identifier
            file_set.add(identifier)

    # Step 3: Compute the intersection of the two sets
    matching_ids = target_set.intersection(file_set)

    # Step 4: Filter the DataFrame based on the matching IDs
    filtered_df = df[df['target'].isin(matching_ids)]

    return filtered_df

In [ ]:
# Example usage
bacteria_ids_file = 'rnadb/mmseqsSilva/bacteria_ids.txt'  # The file created by the bash command
filtered_df = filter_targets_from_file(revi, bacteria_ids_file)

In [ ]:
def plot_unique_targets_by_seq_id(df):
    # Define the sequence identity thresholds
    seq_id_thresholds = np.round(np.arange(0.9, 1.01, 0.01), 2)  # 0.9 to 1.0 inclusive

    # Define the fixed target coverage threshold
    target_cov_threshold = 0.8

    # Define the custom color palette and order
    custom_palette = ['#a1c9f4', '#b9f2f0', '#8de5a1', '#ffb482', '#fab0e4']
    assembler_order = {
        "CarpeDeam\n(safe mode)": 0, 
        "CarpeDeam\n(unsafe mode)": 1, 
        "PenguiN": 2, 
        "MEGAHIT": 3, 
        "metaSPAdes": 4
    }

    # Create a color mapping for assemblers
    assembler_colors = {name: custom_palette[idx] for name, idx in assembler_order.items()}

    # Set seaborn style to whitegrid
    sns.set_style("whitegrid")

    # Loop through each sample
    for sample in df['sample'].unique():
        # Filter data for the current sample
        sample_df = df[df['sample'] == sample]
        
        # Prepare an empty list to collect data for plotting
        plot_data = []

        # Loop through each sequence identity threshold
        for seq_id in seq_id_thresholds:
            # Filter based on seq.Id. and targetCov thresholds
            filtered = sample_df[(sample_df['seq.Id.'] >= seq_id) & (sample_df['targetCov'] >= target_cov_threshold)]
            
            # Count unique targets for each assembler
            counts = filtered.groupby('assembler')['target'].nunique()
            print(seq_id, counts)
            
            # Append the results to the plot data
            for assembler, count in counts.items():
                plot_data.append({
                    'Sequence Identity Threshold (%)': int(seq_id * 100),  # Convert to percentage
                    'Number of Unique Targets': count,
                    'Assembler': assembler
                })

        # Convert plot_data to a DataFrame
        plot_df = pd.DataFrame(plot_data)

        # Sort the assembler column based on the given order
        plot_df['Assembler'] = pd.Categorical(
            plot_df['Assembler'], 
            categories=assembler_order.keys(), 
            ordered=True
        )

        # Plot the data
        fig, ax = plt.subplots(figsize=(6, 4))  # Adjusted plot size
        sns.lineplot(
            data=plot_df,
            x='Sequence Identity Threshold (%)',
            y='Number of Unique Targets',
            hue='Assembler',
            palette=assembler_colors,
            marker='o',
            linewidth=2.75,  # Thicker lines
            ax=ax
        )
        sampleshort = sample.split(".")[0]
        ax.set_title(f"Dataset: {sampleshort}", fontsize=14)
        ax.set_xlabel("Sequence Identity Threshold (%)", fontsize=14)
        ax.set_ylabel("# Unique Hits", fontsize=14)
        ax.tick_params(axis='both', labelsize=12)

        # Force x-ticks to be percentages (90, 91, ..., 100)
        x_ticks = list(range(90, 101))
        ax.set_xticks(x_ticks)

        # Remove the legend from the main plot
        ax.get_legend().remove()

        # Customize legend handles with thicker lines
        legend_handles = [
            Line2D(
                [0], [0], color=assembler_colors[name], lw=3.5, marker='o', markersize=8, label=name
            ) for name in assembler_order.keys()
        ]

        # Add a legend under the plot in two lines
        fig.legend(
            handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.15), 
            fancybox=True, shadow=True, ncol=3, fontsize=12, title_fontsize=12
        )

        plt.tight_layout()
        plt.subplots_adjust(bottom=0.3)  # Adjust space for the legend

        # Save or display the plot
        plt.show()

In [ ]:
plot_unique_targets_by_seq_id(filtered_df)